# Notebook 1: Major Cluster Annotation

This notebook covers initial data loading and major cell type annotation for the BICRC (Biliary Integrated Rectal Cancer) spatial transcriptomics cohort.

**Overview:**
1. Patient metadata and cohort configuration
2. Load integrated AnnData object
3. Define Major_cluster (Epithelial / Stromal / Myeloid / T cell / B cell)
4. Define Major_cluster_pathol (Tumor / Normal / Stromal / Myeloid / T cell / B cell)
5. UMAP visualization by major cluster
6. Cell composition bar plots per sample and timepoint

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import scanpy as sc
import anndata

sc.settings.verbosity = 3
sc.settings.figdir = '../figures/'
plt.rcParams['figure.dpi'] = 150

## 2. Patient Metadata Configuration

BICRC cohort: 24 rectal cancer patients with longitudinal Xenium spatial transcriptomics data (Pre-treatment, JustAfter CRT, and Resection timepoints).

- **NR/SD** (Pt-1 to Pt-8): No Response / Stable Disease
- **MPR** (Pt-9 to Pt-16): Major Pathological Response
- **pCR/CR** (Pt-17 to Pt-24): Pathological Complete Response / Complete Response

JustAfter samples available for: Pt-3, Pt-5, Pt-11, Pt-13 only.

In [ ]:
# ── Cohort configuration ──────────────────────────────────────────────────────

# All sample names (54 total: Pre + Resection for all 24 patients;
# JustAfter only for Pt-3, Pt-5, Pt-11, Pt-13)
sample_name_ls = [
    'Pt-1_Pre',  'Pt-1_Resection',
    'Pt-2_Pre',  'Pt-2_Resection',
    'Pt-3_Pre',  'Pt-3_JustAfter',  'Pt-3_Resection',
    'Pt-4_Pre',  'Pt-4_Resection',
    'Pt-5_Pre',  'Pt-5_JustAfter',  'Pt-5_Resection',
    'Pt-6_Pre',  'Pt-6_Resection',
    'Pt-7_Pre',  'Pt-7_Resection',
    'Pt-8_Pre',  'Pt-8_Resection',
    'Pt-9_Pre',  'Pt-9_Resection',
    'Pt-10_Pre', 'Pt-10_Resection',
    'Pt-11_Pre', 'Pt-11_JustAfter', 'Pt-11_Resection',
    'Pt-12_Pre', 'Pt-12_Resection',
    'Pt-13_Pre', 'Pt-13_JustAfter', 'Pt-13_Resection',
    'Pt-14_Pre', 'Pt-14_Resection',
    'Pt-15_Pre', 'Pt-15_Resection',
    'Pt-16_Pre', 'Pt-16_Resection',
    'Pt-17_Pre', 'Pt-17_Resection',
    'Pt-18_Pre', 'Pt-18_Resection',
    'Pt-19_Pre', 'Pt-19_Resection',
    'Pt-20_Pre', 'Pt-20_Resection',
    'Pt-21_Pre', 'Pt-21_Resection',
    'Pt-22_Pre', 'Pt-22_Resection',
    'Pt-23_Pre', 'Pt-23_Resection',
    'Pt-24_Pre', 'Pt-24_Resection',
]

# Response group assignment
def get_response(sample_name):
    pt_num = int(sample_name.split('_')[0].replace('Pt-', ''))
    if pt_num <= 8:
        return 'SD'
    elif pt_num <= 16:
        return 'MPR'
    else:
        return 'CR'

# Disease-free survival (DFS) vs progression (PD) mapping
# DFS patients: Pt-3, Pt-11, Pt-12, Pt-15, Pt-17 through Pt-24
DFS_patients = {3, 11, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24}

def get_relapse(sample_name):
    pt_num = int(sample_name.split('_')[0].replace('Pt-', ''))
    return 'DFS' if pt_num in DFS_patients else 'PD'

# Color palettes
response_colors = {'SD': 'red', 'MPR': 'orange', 'CR': 'blue'}
timepoint_colors = {'Pre': '#1f77b4', 'JustAfter': '#ff7f0e', 'Resection': '#2ca02c'}

## 3. Load Integrated Data

In [ ]:
# Load the fully integrated and filtered AnnData object
adata = sc.read_h5ad('../data/integrate_adata_filtered.h5ad')
print(adata)
print(adata.obs.columns.tolist())

## 4. Define Major Cluster Categories

Two complementary categorical labels are created:

- **Major_cluster**: Groups all epithelial cells (tumor + normal) into 'Epithelial', and all fibroblast/endothelial cells into 'Stromal'.
- **Major_cluster_pathol**: Distinguishes tumor from normal epithelium, which is essential for pathological analysis.

In [ ]:
# Major_cluster: broad cell type grouping
adata.obs['Major_cluster'] = adata.obs['Cell_type_integrate'].replace({
    'Endothelial': 'Stromal',
    'CAF': 'Stromal',
    'Fibroblast': 'Stromal',
    'Tumor': 'Epithelial',
})

# Major_cluster_pathol: distinguishes tumor from normal epithelium
adata.obs['Major_cluster_pathol'] = adata.obs['Cell_type_integrate'].replace({
    'Endothelial': 'Stromal',
    'CAF': 'Stromal',
    'Fibroblast': 'Stromal',
    'Epithelial': 'Normal',
})

print('Major_cluster categories:', adata.obs['Major_cluster'].unique().tolist())
print('Major_cluster_pathol categories:', adata.obs['Major_cluster_pathol'].unique().tolist())

## 5. UMAP Visualization

In [ ]:
# Color palette for major clusters
major_cluster_colors = {
    'Epithelial': '#e41a1c',
    'Stromal':    '#ff7f00',
    'Myeloid':    '#4daf4a',
    'T cell':     '#377eb8',
    'B cell':     '#984ea3',
}

major_cluster_pathol_colors = {
    'Tumor':   '#e41a1c',
    'Normal':  '#f781bf',
    'Stromal': '#ff7f00',
    'Myeloid': '#4daf4a',
    'T cell':  '#377eb8',
    'B cell':  '#984ea3',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc.pl.umap(
    adata, color='Major_cluster',
    palette=major_cluster_colors,
    ax=axes[0], show=False, title='Major Cluster'
)
sc.pl.umap(
    adata, color='Major_cluster_pathol',
    palette=major_cluster_pathol_colors,
    ax=axes[1], show=False, title='Major Cluster (Pathological)'
)

plt.tight_layout()
plt.savefig('../figures/01_umap_major_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# UMAP colored by sample and timepoint
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc.pl.umap(adata, color='Sample', ax=axes[0], show=False, title='Sample')
sc.pl.umap(adata, color='Timepoint', ax=axes[1], show=False, title='Timepoint')

plt.tight_layout()
plt.savefig('../figures/01_umap_sample_timepoint.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Cell Composition Analysis

Bar plots showing the proportion of each major cell type per sample, grouped by response category and timepoint.

In [ ]:
def plot_cell_composition(adata, cluster_key, sample_order, color_map, title, output_path):
    """Plot stacked bar chart of cell type proportions per sample."""
    composition = (
        adata.obs.groupby(['Sample', cluster_key])
        .size()
        .unstack(fill_value=0)
    )
    composition = composition.div(composition.sum(axis=1), axis=0)
    composition = composition.reindex(sample_order, fill_value=0)

    fig, ax = plt.subplots(figsize=(len(sample_order) * 0.4 + 2, 5))
    composition.plot(
        kind='bar', stacked=True, ax=ax,
        color=[color_map.get(c, 'grey') for c in composition.columns],
        edgecolor='none', width=0.85
    )
    ax.set_title(title)
    ax.set_xlabel('Sample')
    ax.set_ylabel('Cell proportion')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Annotate Response and Timepoint in obs
adata.obs['Response'] = adata.obs['Sample'].apply(get_response)
adata.obs['Timepoint'] = adata.obs['Sample'].apply(lambda s: s.split('_', 1)[1])
adata.obs['Relapse'] = adata.obs['Sample'].apply(get_relapse)

# Ordered sample list (Pre → JustAfter → Resection within each patient)
ordered_samples = [s for s in sample_name_ls if s in adata.obs['Sample'].unique()]

# Plot Major_cluster composition
plot_cell_composition(
    adata, 'Major_cluster', ordered_samples,
    major_cluster_colors,
    'Cell Composition by Major Cluster',
    '../figures/01_composition_major_cluster.png'
)

# Plot Major_cluster_pathol composition
plot_cell_composition(
    adata, 'Major_cluster_pathol', ordered_samples,
    major_cluster_pathol_colors,
    'Cell Composition by Major Cluster (Pathological)',
    '../figures/01_composition_major_cluster_pathol.png'
)

In [ ]:
# Summary statistics: cell counts per major cluster and timepoint
summary = (
    adata.obs.groupby(['Response', 'Timepoint', 'Major_cluster_pathol'])
    .size()
    .reset_index(name='cell_count')
)
print(summary.pivot_table(
    index=['Response', 'Timepoint'],
    columns='Major_cluster_pathol',
    values='cell_count',
    aggfunc='sum'
))

## 7. Save Annotated Object

In [ ]:
# Save with major cluster annotations added
adata.write_h5ad('../data/integrate_adata_annotated.h5ad')
print('Saved integrate_adata_annotated.h5ad')